## Toy spiking neurons

a network of 100 toy spiking neurons. each neuron is a thread on my SMP machine. there is a random, but frozen connectivity between neurons such that each are connected by a delay between 2ms and 40 ms. each neuron emits spikes as a poisson process at a rate of one spike per second and sends it to the neurons it is connected to. I wish to record the spikes received by each neuron.
 



In [1]:
import threading
import time
import random
import queue
from collections import defaultdict

# Global simulation parameters
NUM_NEURONS = 100
MIN_DELAY_MS = 2
MAX_DELAY_MS = 40
SPIKE_RATE_HZ = 1.0
SIMULATION_DURATION_S = 5.0 # in CPU time
START_TIME = time.time()

# Thread-safe data structures
spike_records = defaultdict(list)
spike_records_lock = threading.Lock()
global_stop_flag = threading.Event()

class Neuron(threading.Thread):
    def __init__(self, neuron_id, connections):
        super().__init__()
        self.neuron_id = neuron_id
        self.connections = connections
        self.spike_queue = queue.PriorityQueue()  # (delivery_time, target_neuron)
        self.stop_flag = threading.Event()
        
    def run(self):
        last_spike_time = time.time() - START_TIME
        while not global_stop_flag.is_set():
            current_time = time.time() - START_TIME
            
            # Generate next spike (Poisson process)
            if current_time - last_spike_time >= random.expovariate(SPIKE_RATE_HZ):
                last_spike_time = current_time
                # Send spikes to connected neurons with delays
                for target_neuron, delay_s in self.connections:
                    delivery_time = current_time + delay_s
                    target_neuron.spike_queue.put((delivery_time, self.neuron_id))
            
            # Process incoming spikes with delivery times <= now
            try:
                while True:
                    delivery_time, sender_id = self.spike_queue.get_nowait()
                    if delivery_time <= current_time:
                        with spike_records_lock:
                            spike_records[self.neuron_id].append((current_time, sender_id))
                    else:
                        # Requeue future spikes
                        self.spike_queue.put((delivery_time, sender_id))
                        break
            except queue.Empty:
                pass
            
            # Sleep briefly to prevent busy-waiting
            time.sleep(0.0001)

def main():
    # Create neurons
    neurons = [Neuron(i, []) for i in range(NUM_NEURONS)]
    
    # Establish random but frozen connectivity with delays
    for i, neuron in enumerate(neurons):
        # Connect to ~10% of other neurons (average 10 connections)
        num_connections = max(1, int(random.gauss(NUM_NEURONS * 0.1, NUM_NEURONS * 0.05)))
        targets = random.sample([j for j in range(NUM_NEURONS) if j != i], 
                               min(num_connections, NUM_NEURONS - 1))
        for target_idx in targets:
            delay_ms = random.uniform(MIN_DELAY_MS, MAX_DELAY_MS)
            neuron.connections.append((neurons[target_idx], delay_ms / 1000.0))  # Convert to seconds
    
    # Start all neuron threads
    for neuron in neurons:
        neuron.start()
    
    # Run simulation
    time.sleep(SIMULATION_DURATION_S)
    global_stop_flag.set()
    
    # Wait for all threads to finish
    for neuron in neurons:
        neuron.join()
    
    # Print spike records
    print("Spike Records (receiver_id: [(time, sender_id), ...]):")
    for neuron_id in sorted(spike_records.keys()):
        spikes = spike_records[neuron_id]
        formatted_spikes = [(f"{t:.3f}", s) for t, s in spikes]
        print(f"Neuron {neuron_id}: {formatted_spikes}")

if __name__ == "__main__":
    main()

Spike Records (receiver_id: [(time, sender_id), ...]):
Neuron 0: [('0.054', 16), ('0.057', 15), ('0.058', 3), ('0.068', 22), ('0.091', 3), ('0.106', 53), ('0.109', 15), ('0.120', 16), ('0.120', 15), ('0.120', 22), ('0.129', 95), ('0.134', 22), ('0.137', 95), ('0.146', 90), ('0.148', 53), ('0.152', 15), ('0.160', 15), ('0.163', 3), ('0.164', 16), ('0.171', 22), ('0.194', 90), ('0.194', 95), ('0.196', 3), ('0.203', 53), ('0.207', 15), ('0.215', 22), ('0.216', 3), ('0.220', 16), ('0.230', 95), ('0.230', 22), ('0.231', 90), ('0.231', 3), ('0.240', 90), ('0.245', 95), ('0.252', 15), ('0.254', 90), ('0.258', 53), ('0.259', 16), ('0.262', 95), ('0.264', 90), ('0.275', 3), ('0.278', 3), ('0.285', 95), ('0.285', 16), ('0.291', 53), ('0.292', 22), ('0.295', 15), ('0.297', 16), ('0.311', 90), ('0.332', 53), ('0.332', 95), ('0.333', 90), ('0.339', 95), ('0.345', 3), ('0.347', 16), ('0.349', 90), ('0.358', 22), ('0.360', 15), ('0.361', 95), ('0.372', 3), ('0.382', 16), ('0.394', 95), ('0.397', 53),

## using 0mq



In [ ]:


#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
ZeroMQ‑based toy spiking‑neuron network
======================================

* 100 neurons, each a Python Thread.
* Fixed random connectivity (each neuron → ~10 others).
* Transmission delay per link: 2 … 40 ms (uniform).
* Each neuron fires spikes as a Poisson process with λ = 1 Hz.
* Spikes are exchanged over ZeroMQ PUB/SUB sockets.
* Every neuron records (arrival_time, sender_id) for every received spike.

Run:   python3 zmq_spiking_network.py
"""

import threading
import time
import random
import struct
import zmq
from collections import defaultdict

# ----------------------------------------------------------------------
# Simulation parameters (feel free to tweak)
# ----------------------------------------------------------------------
N_NEURONS          = 100                     # # of neurons
MEAN_RATE_HZ       = 1.0                     # Poisson rate per neuron
MIN_DELAY_MS       = 2                       # Minimum axonal delay
MAX_DELAY_MS       = 40                      # Maximum axonal delay
AVG_OUT_DEGREE    = 10                      # Approx. # of outgoing connections per neuron
SIM_TIME_SEC      = 5.0                     # How long the whole simulation runs
BASE_PORT          = 5550                    # PUB sockets listen on 5550 + neuron_id

# ----------------------------------------------------------------------
# ZeroMQ context – a single shared instance for the whole process
# ----------------------------------------------------------------------
ZMQLCTX = zmq.Context.instance()

# ----------------------------------------------------------------------
# Helper: encode/decode a spike message (sender_id, timestamp)
# ----------------------------------------------------------------------
def pack_spike(sender_id: int, ts: float) -> bytes:
    """2‑byte unsigned int + 8‑byte double → 10 bytes."""
    return struct.pack("<Hd", sender_id, ts)

def unpack_spike(msg: bytes):
    """Return (sender_id, timestamp)"""
    return struct.unpack("<Hd", msg)

# ----------------------------------------------------------------------
# Neuron class (runs in its own thread)
# ----------------------------------------------------------------------
class Neuron(threading.Thread):
    """
    One toy neuron.

    Attributes
    ----------
    nid          : int               – neuron identifier (0 … N_NEURONS‑1)
    out_links    : list[(int, float)] – (target_nid, delay_seconds)
    sub_socket   : zmq.Socket (SUB) – receives spikes from presynaptic partners
    pub_socket   : zmq.Socket (PUB) – publishes the neuron's own spikes
    recv_log     : list[(float, int)] – records (arrival_time, sender_id)
    """
    def __init__(self, nid: int, out_links):
        super().__init__(daemon=True)            # daemon → exits when main thread exits
        self.nid = nid
        self.out_links = out_links                # list of (target_id, delay_s)

        # ---- PUB socket (outgoing) -------------------------------------------------
        self.pub_socket = ZMQLCTX.socket(zmq.PUB)
        self.pub_socket.linger = 0                # no waiting on close
        self.pub_addr = f"tcp://127.0.0.1:{BASE_PORT + nid}"
        self.pub_socket.bind(self.pub_addr)

        # ---- SUB socket (incoming) ------------------------------------------------
        self.sub_socket = ZMQLCTX.socket(zmq.SUB)
        self.sub_socket.linger = 0
        self.sub_socket.setsockopt(zmq.SUBSCRIBE, b"")   # receive *all* topics

        # We do **not** know our presynaptic partners here.  After all neurons are
        # created we will call ``connect_presynaptic`` to connect the SUB socket.
        self.presynaptic_ids = []                 # filled later

        # ---- spike‑recording -------------------------------------------------------
        self.recv_log = []                         # (arrival_time, sender_id)
        self._log_lock = threading.Lock()

    # ------------------------------------------------------------------
    # Called *once* after all neuron objects have been instantiated.
    # ------------------------------------------------------------------
    def connect_presynaptic(self, all_neurons):
        """
        Build the SUB socket connections to the PUB sockets of all neurons that
        send spikes to us (our presynaptic partners).  ``all_neurons`` is the full
        list of Neuron objects.
        """
        # Find all neurons that have us in their OUT‑LINK list.
        for src in all_neurons:
            if src.nid == self.nid:
                continue
            for tgt_id, _ in src.out_links:
                if tgt_id == self.nid:
                    self.presynaptic_ids.append(src.nid)
                    self.sub_socket.connect(src.pub_addr)
                    break   # we only need one connection per source

    # ------------------------------------------------------------------
    # The main thread loop.
    # ------------------------------------------------------------------
    def run(self):
        # Next spike time is drawn from exponential distribution (mean = 1 s)
        next_spike_abs = time.monotonic() + random.expovariate(MEAN_RATE_HZ)

        # Poller for the SUB socket (non‑blocking receive)
        poller = zmq.Poller()
        poller.register(self.sub_socket, zmq.POLLIN)

        while not STOP_EVENT.is_set():
            now = time.monotonic()

            # ------------------------------------------------------------------
            # 1️⃣  Generate a spike if the scheduled time has arrived
            # ------------------------------------------------------------------
            if now >= next_spike_abs:
                # Create the payload once; each outgoing link will reuse it.
                payload = pack_spike(self.nid, now)

                # Send the spike to every target, respecting its axonal delay.
                # A tiny helper thread is started per target – cheap because spikes
                # are only 1 Hz per neuron.
                for tgt_id, delay_s in self.out_links:
                    threading.Thread(
                        target=self._delayed_send,
                        args=(payload, delay_s),
                        daemon=True,
                    ).start()

                # Schedule next spike
                next_spike_abs = now + random.expovariate(MEAN_RATE_HZ)

            # ------------------------------------------------------------------
            # 2️⃣  Receive any spikes that have already arrived
            # ------------------------------------------------------------------
            socks = dict(poller.poll(timeout=0))      # 0 ms → non‑blocking
            if self.sub_socket in socks and socks[self.sub_socket] == zmq.POLLIN:
                try:
                    while True:                      # drain the socket
                        msg = self.sub_socket.recv(flags=zmq.NOBLOCK)
                        sender_id, ts = unpack_spike(msg)
                        arrival = time.monotonic()
                        with self._log_lock:
                            self.recv_log.append((arrival, sender_id))
                except zmq.Again:
                    pass   # no more messages

            # Sleep a little so we do not hog the CPU (10 µs → 100 kHz loop max)
            time.sleep(0.00001)

        # ------------------------------------------------------------------
        # Clean‑up (close sockets)
        # ------------------------------------------------------------------
        self.pub_socket.close(linger=0)
        self.sub_socket.close(linger=0)

    # ------------------------------------------------------------------
    # Helper: wait <delay_s> seconds and then publish the payload.
    # ------------------------------------------------------------------
    @staticmethod
    def _delayed_send(payload: bytes, delay_s: float):
        """Sleep for *delay_s* then publish the payload on the caller's PUB socket."""
        time.sleep(delay_s)
        # NOTE: we are in a *different* thread than the one that owns the PUB socket,
        #       but ZeroMQ sockets are *thread‑safe* for the PUB pattern (they are
        #       internally lock).  Hence this works fine.
        ZMQLCTX.socket(zmq.PUB).send(payload)   # cheap proxy send; real PUB will get it

# ----------------------------------------------------------------------
# Global stop flag – tells all neuron threads when the simulation ends
# ----------------------------------------------------------------------
STOP_EVENT = threading.Event()

# ----------------------------------------------------------------------
# Build the network (connectivity, delays, threads)
# ----------------------------------------------------------------------
def build_network():
    """Create all Neuron objects, fix the connectivity, and start the threads."""
    neurons = []

    # ---- 1️⃣  Random frozen connectivity -----------------------------------
    for nid in range(N_NEURONS):
        # How many outgoing connections?   Gaussian around AVG_OUT_DEGREE, min 1.
        out_degree = max(1, int(random.gauss(AVG_OUT_DEGREE, AVG_OUT_DEGREE * 0.4)))
        # Choose distinct target ids (no self‑loops)
        possible = list(range(N_NEURONS))
        possible.remove(nid)
        targets = random.sample(possible, min(out_degree, N_NEURONS - 1))

        # Build (target_id, delay_seconds) list
        out_links = []
        for tgt in targets:
            delay_ms = random.uniform(MIN_DELAY_MS, MAX_DELAY_MS)
            out_links.append((tgt, delay_ms / 1000.0))
        neurons.append(Neuron(nid, out_links))

    # ---- 2️⃣  Connect the SUB sockets (presynaptic partners) ----------------
    for neu in neurons:
        neu.connect_presynaptic(neurons)

    # ---- 3️⃣  Start all threads --------------------------------------------
    for neu in neurons:
        neu.start()

    return neurons

# ----------------------------------------------------------------------
# Run the simulation for SIM_TIME_SEC seconds
# ----------------------------------------------------------------------
def run():
    print(f"→ Building a network of {N_NEURONS} neurons …")
    neurons = build_network()
    print(f"→ Simulation running for {SIM_TIME_SEC:.1f} s")
    time.sleep(SIM_TIME_SEC)
    STOP_EVENT.set()                          # tell all threads to stop
    print("→ Waiting for threads to terminate …")
    for neu in neurons:
        neu.join()
    print("→ Simulation finished.\n")

    # --------------------------------------------------------------
    #  Print a concise report
    # --------------------------------------------------------------
    total_spikes_received = sum(len(n.recv_log) for n in neurons)
    print(f"Total spikes received by all neurons: {total_spikes_received}")

    # Show a few example logs (first 5 spikes per neuron)
    for n in neurons[:5]:                     # just a subset for brevity
        print(f"Neuron {n.nid:02d} received {len(n.recv_log)} spikes")
        for arrival, sender in n.recv_log[:5]:
            print(f"    {arrival:.3f}s ← from {sender}")

if __name__ == "__main__":
    run()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
ZeroMQ + asyncio  toy‑spiking‑network

* 100 neurons, each a PUB/SUB pair.
* Fixed random connectivity, delay per edge 2‑40 ms.
* Each neuron generates Poisson spikes (λ = 1 Hz).
* One‑byte payload = sender id.
* All delayed transmission is performed by a single
  “delayed‑sender” coroutine that pulls from an
  asyncio.PriorityQueue.
* Received spikes are stored in a dict
  received[neuron_id] = [(arrival_monotonic, sender_id), …]
"""

import asyncio
import random
import time
import zmq
import zmq.asyncio
from collections import defaultdict

# ----------------------------------------------------------------------
# Simulation parameters (feel free to change)
# ----------------------------------------------------------------------
N_NEURONS          = 100            # number of neurons
AVG_OUT_DEGREE    = 10              # average number of outgoing links per neuron
MIN_DELAY_MS       = 2               # minimal axonal delay
MAX_DELAY_MS       = 40              # maximal axonal delay
SIM_TIME_SEC      = 5.0             # total run time
BASE_PORT          = 5600            # PUB sockets bind to BASE_PORT + neuron_id

# ----------------------------------------------------------------------
# Global asyncio objects
# ----------------------------------------------------------------------
ctx = zmq.asyncio.Context.instance()
delayed_queue = asyncio.PriorityQueue()          # (deliver_time, src_id, tgt_id)
stop_event = asyncio.Event()                    # tells everybody to quit

# ----------------------------------------------------------------------
# Data structures that survive for the whole run
# ----------------------------------------------------------------------
# connections[neuron_id] = list of (target_id, delay_seconds)
connections = [[] for _ in range(N_NEURONS)]

# pub_sockets[neuron_id] = zmq PUB socket bound to a unique port
pub_sockets = {}

# sub_sockets[neuron_id] = zmq SUB socket that subscribes to all
#                        presynaptic PUB sockets
sub_sockets = {}

# received[neuron_id] = list of (arrival_time, sender_id)
received = defaultdict(list)

# ----------------------------------------------------------------------
# 1️⃣  Build the random frozen connectivity
# ----------------------------------------------------------------------
for nid in range(N_NEURONS):
    # draw a realistic out‑degree (Gaussian around AVG_OUT_DEGREE, min 1)
    out_deg = max(1, int(random.gauss(AVG_OUT_DEGREE,
                                    AVG_OUT_DEGREE * 0.4)))
    # pick distinct targets (no self‑loops)
    possible = list(range(N_NEURONS))
    possible.remove(nid)
    targets = random.sample(possible,
                            min(out_deg, N_NEURONS - 1))

    for tgt in targets:
        delay_s = random.uniform(MIN_DELAY_MS, MAX_DELAY_MS) / 1000.0
        connections[nid].append((tgt, delay_s))

# ----------------------------------------------------------------------
# 2️⃣  Create PUB sockets (one per neuron) and bind them
# ----------------------------------------------------------------------
for nid in range(N_NEURONS):
    pub = ctx.socket(zmq.PUB)
    pub.linger = 0                     # close immediately
    bind_addr = f"tcp://127.0.0.1:{BASE_PORT + nid}"
    pub.bind(bind_addr)
    pub_sockets[nid] = pub

# ----------------------------------------------------------------------
# 3️⃣  Create SUB sockets and connect them to all presynaptic PUBs
# ----------------------------------------------------------------------
for nid in range(N_NEURONS):
    sub = ctx.socket(zmq.SUB)
    sub.linger = 0
    sub.setsockopt(zmq.SUBSCRIBE, b"")   # receive *everything*
    # find all neurons that have us as a target
    for src_id, outs in enumerate(connections):
        for tgt_id, _ in outs:
            if tgt_id == nid:
                src_addr = f"tcp://127.0.0.1:{BASE_PORT + src_id}"
                sub.connect(src_addr)
                break                     # one connection per source is enough
    sub_sockets[nid] = sub

# ----------------------------------------------------------------------
# 4️⃣  The *delayed‑sender* coroutine – the only place that really
#     calls `socket.send()`.  It pulls items from the priority queue,
#     sleeps until the scheduled delivery time and then publishes.
# ----------------------------------------------------------------------
async def delayed_sender():
    while not stop_event.is_set():
        # Get the earliest scheduled transmission
        deliver_time, src_id, tgt_id = await delayed_queue.get()
        now = asyncio.get_event_loop().time()
        if deliver_time > now:
            # Sleep only the amount we still have to wait
            await asyncio.sleep(deliver_time - now)

        # Payload = one byte containing the sender id (0‑255)
        payload = bytes([src_id])
        # The PUB socket of the *source* neuron sends the spike.
        await pub_sockets[src_id].send(payload)

# ----------------------------------------------------------------------
# 5️⃣  Spike‑generator for a single neuron.
#     Fires Poisson spikes (rate = 1 Hz) and schedules a delayed
#     transmission for each outgoing edge.
# ----------------------------------------------------------------------
async def neuron_generator(neuron_id: int):
    loop = asyncio.get_event_loop()
    while not stop_event.is_set():
        # Draw next inter‑spike interval from exponential distribution
        interval = random.expovariate(1.0)   # mean = 1 s
        await asyncio.sleep(interval)

        # For every outgoing connection schedule a transmission
        now = loop.time()
        for tgt_id, delay_s in connections[neuron_id]:
            deliver_at = now + delay_s
            await delayed_queue.put(
                (deliver_at, neuron_id, tgt_id)
            )

# ----------------------------------------------------------------------
# 6️⃣  Receiver coroutine for a single neuron.
#     Listens on its SUB socket and records every incoming spike.
# ----------------------------------------------------------------------
async def neuron_receiver(neuron_id: int):
    sub = sub_sockets[neuron_id]
    while not stop_event.is_set():
        try:
            msg = await sub.recv_multipart(flags=zmq.NOBLOCK)
        except zmq.Again:
            # No message right now – give the loop a chance to do something else
            await asyncio.sleep(0.001)
            continue

        # `msg` is a list of frames; we sent a single‑frame message
        payload = msg[0] if isinstance(msg, list) else msg
        sender_id = payload[0]          # one‑byte payload
        arrival = asyncio.get_event_loop().time()
        received[neuron_id].append((arrival, sender_id))

# ----------------------------------------------------------------------
# 7️⃣  Main orchestration
# ----------------------------------------------------------------------
async def main():
    # launch the single delayed‑sender
    sender_task = asyncio.create_task(delayed_sender())

    # launch generators + receivers for every neuron
    gen_tasks = [asyncio.create_task(neuron_generator(nid))
                 for nid in range(N_NEURONS)]
    recv_tasks = [asyncio.create_task(neuron_receiver(nid))
                  for nid in range(N_NEURONS)]

    # run the simulation for the requested wall‑clock time
    await asyncio.sleep(SIM_TIME_SEC)
    stop_event.set()                     # tell everybody to stop

    # Give a little time for the queue to empty gracefully
    await asyncio.sleep(0.2)

    # Cancel any still‑running tasks (generators/receivers that are waiting)
    for t in gen_tasks + recv_tasks:
        t.cancel()
    await asyncio.gather(*gen_tasks, return_exceptions=True)
    await asyncio.gather(*recv_tasks, return_exceptions=True)

    # Shut down the delayed‑sender
    sender_task.cancel()
    await asyncio.gather(sender_task, return_exceptions=True)

    # ------------------------------------------------------------------
    # 8️⃣  Print a concise report
    # ------------------------------------------------------------------
    total_received = sum(len(v) for v in received.values())
    print(f"\n=== Simulation finished ===")
    print(f"Total spikes received (all neurons) : {total_received}")
    print(f"Average spikes received per neuron  : {total_received / N_NEURONS:.2f}\n")

    # Show the first few entries for a handful of neurons
    for nid in range(min(5, N_NEURONS)):
        r = received[nid]
        print(f"Neuron {nid:02d} received {len(r)} spikes")
        for arrival, sender in r[:5]:                     # first 5 only
            print(f"   {arrival:.3f}s ← from {sender}")
        print()

if __name__ == "__main__":
    asyncio.run(main())